In [ ]:
# **Objective:** Explain model predictions using SHAP values
# 
# **Key analyses:**
# 1. Global feature importance
# 2. Local explanations for individual predictions
# 3. Feature interaction effects
# 4. SHAP dependence plots
# 5. Business insights from explanations


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# %%
# Load data and model
print("=" * 80)
print("LOADING MODEL AND DATA")
print("=" * 80)

X = pd.read_csv('../data/processed/features_scaled.csv')
y = pd.read_csv('../data/processed/target.csv')['Churn']

# Load best model
model_files = [f for f in os.listdir('../models') if f.startswith('best_model_')]
if model_files:
    best_model_path = f"../models/{model_files[0]}"
    if best_model_path.endswith('.pkl'):
        model = joblib.load(best_model_path)
        print(f"✅ Loaded model: {best_model_path}")
        print(f"   Model type: {type(model).__name__}")
    else:
        print("❌ Neural network model - SHAP may not work well")
        exit()
else:
    print("❌ No model found! Please run model training notebook first.")
    exit()

# %%
# ==================== SMART SHAP EXPLAINER SELECTION ====================
print("\n" + "=" * 80)
print("SELECTING APPROPRIATE SHAP EXPLAINER")
print("=" * 80)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

model_type = type(model).__name__
print(f"\n🔍 Detected Model Type: {model_type}")

# Select appropriate explainer based on model type
if model_type in ['RandomForestClassifier', 'XGBClassifier', 'LGBMClassifier', 
                  'GradientBoostingClassifier', 'ExtraTreesClassifier']:
    print("📊 Model is tree-based → Using TreeExplainer (FASTEST)")
    explainer = shap.TreeExplainer(model)
    explainer_type = "tree"
    
elif model_type in ['LogisticRegression', 'LinearRegression', 'SGDClassifier']:
    print("📊 Model is linear → Using LinearExplainer")
    # Use background data for LinearExplainer
    background = X.sample(min(1000, len(X)), random_state=42)
    explainer = shap.LinearExplainer(model, background)
    explainer_type = "linear"
    
elif model_type in ['MLPClassifier', 'MLPRegressor']:
    print("📊 Model is neural network → Using DeepExplainer")
    background = X.sample(min(100, len(X)), random_state=42)
    explainer = shap.DeepExplainer(model, background)
    explainer_type = "deep"
    
else:
    print("📊 Model type not recognized → Using KernelExplainer (SLOWEST)")
    # KernelExplainer works for ANY model but is slow
    background = X.sample(min(50, len(X)), random_state=42)
    explainer = shap.KernelExplainer(model.predict_proba, background)
    explainer_type = "kernel"

print(f"\n✅ SHAP explainer created successfully!")
print(f"   Explainer type: {explainer_type.upper()}")
print("=" * 80)
# ==============================================================================

# %%
# Sample data for explanation
print("\n📦 PREPARING DATA SAMPLE")
print("=" * 80)

sample_size = min(1000, len(X))
X_sample = X.sample(n=sample_size, random_state=42)

print(f"Total dataset size: {len(X):,}")
print(f"Sample size for SHAP: {sample_size:,}")
print(f"Features: {X_sample.shape[1]}")

# %%
# Calculate SHAP values
print("\n⏳ CALCULATING SHAP VALUES")
print("=" * 80)
print("   (This may take 1-5 minutes depending on model and sample size)")

if explainer_type == "kernel":
    # KernelExplainer is very slow - use smaller sample
    X_sample_small = X_sample.sample(min(100, len(X_sample)), random_state=42)
    print(f"   Using smaller sample for KernelExplainer: {len(X_sample_small)}")
    shap_values = explainer.shap_values(X_sample_small)
    X_sample = X_sample_small  # Update to match
else:
    shap_values = explainer.shap_values(X_sample)

print(f"✅ SHAP values calculated!")
print(f"   Type: {type(shap_values)}")

# Handle different SHAP output formats
if isinstance(shap_values, list):
    if len(shap_values) == 2:
        # Binary classification: [negative_class, positive_class]
        shap_values_for_plot = shap_values[1]  # Positive class (churn)
        print(f"   Binary classification → Using positive class values")
    else:
        shap_values_for_plot = shap_values[0]
else:
    shap_values_for_plot = shap_values

print(f"   Final shape: {np.array(shap_values_for_plot).shape}")
print("=" * 80)

# %%
# ==================== GLOBAL EXPLANATIONS ====================
print("\n📊 GENERATING GLOBAL EXPLANATIONS")
print("=" * 80)

# 1. Summary plot
print("\n1. Global Feature Importance (Summary Plot)")
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values_for_plot, X_sample, show=False, plot_size=(12, 10))
plt.title('Global Feature Importance', fontsize=16, fontweight='bold')
plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()
print("   ✅ Saved to: ../reports/figures/shap_summary.png")

# %%
# 2. Feature importance bar chart
print("\n2. Feature Importance Bar Chart")
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values_for_plot, X_sample, plot_type="bar", show=False, plot_size=(12, 10))
plt.title('Feature Importance (Mean Absolute SHAP)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/shap_importance_bar.png', dpi=300, bbox_inches='tight')
plt.show()
print("   ✅ Saved to: ../reports/figures/shap_importance_bar.png")

# %%
# 3. SHAP values distribution
print("\n3. SHAP Values Distribution")
shap_df = pd.DataFrame(shap_values_for_plot, columns=X_sample.columns)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)

plt.figure(figsize=(14, 10))
mean_abs_shap.head(20).plot(kind='barh')
plt.title('Top 20 Features by Mean Absolute SHAP', fontsize=16, fontweight='bold')
plt.xlabel('Mean Absolute SHAP Value')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/shap_mean_abs.png', dpi=300, bbox_inches='tight')
plt.show()
print("   ✅ Saved to: ../reports/figures/shap_mean_abs.png")

# %%
# ==================== LOCAL EXPLANATIONS ====================

# 4. Individual prediction explanations
print("\n4. Local Explanations for Individual Predictions")

# Select some interesting cases
high_churn_prob_idx = np.argsort(shap_values.sum(axis=1))[-3:]  # Top 3 highest churn probability
low_churn_prob_idx = np.argsort(shap_values.sum(axis=1))[:3]   # Top 3 lowest churn probability

# Plot for high churn probability customers
for idx in high_churn_prob_idx:
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[idx],
            base_values=explainer.expected_value,
            data=X_sample.iloc[idx].values,
            feature_names=X_sample.columns.tolist()
        ),
        max_display=10
    )
    plt.title(f'Customer {idx} - High Churn Risk', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'../reports/figures/shap_waterfall_high_{idx}.png', dpi=300, bbox_inches='tight')
    plt.show()

# Plot for low churn probability customers
for idx in low_churn_prob_idx:
    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_values[idx],
            base_values=explainer.expected_value,
            data=X_sample.iloc[idx].values,
            feature_names=X_sample.columns.tolist()
        ),
        max_display=10
    )
    plt.title(f'Customer {idx} - Low Churn Risk', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'../reports/figures/shap_waterfall_low_{idx}.png', dpi=300, bbox_inches='tight')
    plt.show()

# %%
# 5. Force plot for specific predictions
print("\n5. Force Plots for Specific Predictions")

# High risk customer
plt.figure(figsize=(20, 3))
shap.force_plot(
    explainer.expected_value,
    shap_values[high_churn_prob_idx[0]],
    X_sample.iloc[high_churn_prob_idx[0]],
    matplotlib=True,
    show=False
)
plt.title('Force Plot - High Churn Risk Customer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/shap_force_high.png', dpi=300, bbox_inches='tight')
plt.show()

# Low risk customer
plt.figure(figsize=(20, 3))
shap.force_plot(
    explainer.expected_value,
    shap_values[low_churn_prob_idx[0]],
    X_sample.iloc[low_churn_prob_idx[0]],
    matplotlib=True,
    show=False
)
plt.title('Force Plot - Low Churn Risk Customer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/shap_force_low.png', dpi=300, bbox_inches='tight')
plt.show()

# %%
# ==================== FEATURE DEPENDENCE ====================

# 6. SHAP dependence plots for top features
print("\n6. SHAP Dependence Plots")

top_features = mean_abs_shap.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    shap.dependence_plot(
        feature,
        shap_values,
        X_sample,
        ax=axes[i],
        show=False,
        interaction_index='auto'
    )
    axes[i].set_title(f'SHAP Dependence - {feature}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/shap_dependence.png', dpi=300, bbox_inches='tight')
plt.show()

# %%
# ==================== BUSINESS INSIGHTS ====================

# 7. Feature contribution analysis
print("\n" + "=" * 80)
print("BUSINESS INSIGHTS FROM SHAP ANALYSIS")
print("=" * 80)

print("\nTop 10 Churn Drivers (Features that increase churn probability):")
top_drivers = mean_abs_shap.head(10)
for rank, (feature, importance) in enumerate(top_drivers.items(), 1):
    # Get average SHAP value (not absolute) to see direction
    avg_shap = shap_df[feature].mean()
    direction = "increases" if avg_shap > 0 else "decreases"
    print(f"  {rank}. {feature}: {importance:.4f} ({direction} churn)")

print("\nKey Insights:")
print("  1. Contract type is likely the strongest predictor")
print("  2. Tenure has significant impact (newer customers churn more)")
print("  3. Monthly charges correlate with churn risk")
print("  4. Payment method affects churn probability")
print("  5. Service bundle complexity may drive churn")

# %%
# 8. Segment-specific insights
print("\nSegment-Specific Insights:")

# High-value customers (high monthly charges)
high_value_mask = X_sample['monthly_charges'] > X_sample['monthly_charges'].quantile(0.75)
high_value_shap = pd.DataFrame(shap_values[high_value_mask], columns=X_sample.columns)
high_value_importance = high_value_shap.abs().mean().sort_values(ascending=False).head(5)

print("  High-Value Customers (>75th percentile charges):")
print("    Top churn drivers:")
for feature, importance in high_value_importance.items():
    print(f"      - {feature}: {importance:.4f}")

# New customers (low tenure)
new_customer_mask = X_sample['tenure'] < X_sample['tenure'].quantile(0.25)
new_customer_shap = pd.DataFrame(shap_values[new_customer_mask], columns=X_sample.columns)
new_customer_importance = new_customer_shap.abs().mean().sort_values(ascending=False).head(5)

print("\n  New Customers (<25th percentile tenure):")
print("    Top churn drivers:")
for feature, importance in new_customer_importance.items():
    print(f"      - {feature}: {importance:.4f}")

# %%
# 9. Generate SHAP report DataFrame
print("\nGenerating SHAP report...")

shap_report = pd.DataFrame({
    'feature': X_sample.columns,
    'mean_shap': shap_df.mean(),
    'mean_abs_shap': shap_df.abs().mean(),
    'std_shap': shap_df.std(),
    'shap_positive_pct': (shap_df > 0).mean() * 100
})

shap_report = shap_report.sort_values('mean_abs_shap', ascending=False)
shap_report.to_csv('../reports/shap_analysis_report.csv', index=False)

print("✅ SHAP report saved to ../reports/shap_analysis_report.csv")

# Display top features
print("\nTop 20 Features by SHAP:")
display(shap_report.head(20))

# %%
# 10. Save SHAP explainer for later use
import pickle

shap_artifacts = {
    'explainer': explainer,
    'shap_values': shap_values,
    'X_sample': X_sample,
    'feature_importance': shap_report
}

with open('../models/shap_explainer.pkl', 'wb') as f:
    pickle.dump(shap_artifacts, f)

print("\n✅ SHAP artifacts saved to ../models/shap_explainer.pkl")

# %%
print("\n" + "=" * 80)
print("SHAP EXPLAINABILITY ANALYSIS COMPLETED!")
print("=" * 80)
print("\nGenerated artifacts:")
print("  - SHAP summary plots (../reports/figures/)")
print("  - Feature importance charts")
print("  - Local explanation plots for individual customers")
print("  - Dependence plots for top features")
print("  - SHAP analysis report (CSV)")
print("  - SHAP explainer object (pickle)")
print("\n" + "=" * 80)

d:\VT\Telecom-Customer-Churn-Prediction-using-Machine-Learning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded model: ../models/best_model_logistic_regression.pkl
Creating SHAP explainer...


InvalidModelError: Model type not yet supported by TreeExplainer: <class 'sklearn.linear_model._logistic.LogisticRegression'>